# ROC Curve and AUC Comparison

This notebook loads the trained DenseNet121, VGG16, and ResNet50 PyTorch models, evaluates them on the shared test split, and plots ROC curves with AUC values.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
import random
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import DenseNet121_Weights, ResNet50_Weights, VGG16_Weights

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.environ.setdefault('TORCH_HOME', '/data/ramialle/research_training/Research_Training/.torch')

DATASET_DIR = Path('/data/ramialle/datasets2/Dataset_BUSI_with_GT_Clean')
IMAGE_DIR = DATASET_DIR / 'images-20260602T203626Z-3-001' / 'images'
TEST_XLSX = DATASET_DIR / 'test.xlsx'
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
NUM_WORKERS = 0
CLASS_NAMES = {0: 'malignant', 1: 'benign'}
POSITIVE_CLASS_ID = 1
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using images from: {IMAGE_DIR}')
print(f'Using device: {DEVICE}')

## Checkpoint Paths

Update these paths if your saved model files use different names. Each checkpoint should contain either a raw PyTorch `state_dict`, `state_dict`, or `model_state_dict`.

In [ ]:
CHECKPOINTS = {
    'DenseNet121': Path('model_checkpoints/densenet121_best.pth'),
    'VGG16': Path('model_checkpoints/vgg16_best.pth'),
    'ResNet50': Path('model_checkpoints/resnet50_best.pth'),
}

missing_checkpoints = [str(path) for path in CHECKPOINTS.values() if not path.exists()]
if missing_checkpoints:
    print('Missing checkpoint files:')
    for path in missing_checkpoints:
        print(f'  - {path}')
    print('\nSave each trained model to the matching path, or edit CHECKPOINTS before running the ROC cells.')

In [ ]:
def _column_index(cell_reference):
    letters = ''.join(ch for ch in cell_reference if ch.isalpha())
    index = 0
    for char in letters:
        index = index * 26 + (ord(char.upper()) - ord('A') + 1)
    return index - 1


def read_xlsx_rows(path):
    namespace = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    with ZipFile(path) as workbook_zip:
        shared_strings = []
        if 'xl/sharedStrings.xml' in workbook_zip.namelist():
            root = ET.fromstring(workbook_zip.read('xl/sharedStrings.xml'))
            for item in root.findall('main:si', namespace):
                text_parts = [node.text or '' for node in item.findall('.//main:t', namespace)]
                shared_strings.append(''.join(text_parts))

        sheet = ET.fromstring(workbook_zip.read('xl/worksheets/sheet1.xml'))
        raw_rows = []
        for row in sheet.findall('.//main:sheetData/main:row', namespace):
            values = []
            for cell in row.findall('main:c', namespace):
                column = _column_index(cell.attrib.get('r', 'A1'))
                while len(values) <= column:
                    values.append('')

                cell_type = cell.attrib.get('t')
                value_node = cell.find('main:v', namespace)
                inline_node = cell.find('main:is/main:t', namespace)
                if cell_type == 's' and value_node is not None:
                    value = shared_strings[int(value_node.text)]
                elif cell_type == 'inlineStr' and inline_node is not None:
                    value = inline_node.text or ''
                elif value_node is not None:
                    value = value_node.text or ''
                else:
                    value = ''
                values[column] = value
            raw_rows.append(values)

    header = [str(value).strip() for value in raw_rows[0]]
    rows = []
    for row in raw_rows[1:]:
        row = row + [''] * (len(header) - len(row))
        record = dict(zip(header, row))
        if record.get('Image'):
            record['Image'] = str(record['Image']).strip()
            record['Label'] = int(float(record['Label']))
            record['path'] = IMAGE_DIR / record['Image']
            record['class_name'] = CLASS_NAMES[record['Label']]
            rows.append(record)
    return rows


test_rows = read_xlsx_rows(TEST_XLSX)
missing_images = [row['path'] for row in test_rows if not row['path'].exists()]
if missing_images:
    raise FileNotFoundError(f'Test split has missing images. First missing file: {missing_images[0]}')

test_labels = np.array([row['Label'] for row in test_rows])
test_counts = {CLASS_NAMES[label]: int(np.sum(test_labels == label)) for label in sorted(CLASS_NAMES)}
print(f'Testing samples: {len(test_rows)}')
print('Test class counts:', test_counts)

In [ ]:
class BreastTumorDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows = rows
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        image = Image.open(row['path']).convert('RGB')
        image = self.transform(image)
        label = torch.tensor(row['Label'], dtype=torch.long)
        return image, label


def build_test_loader(weights):
    imagenet_normalize = weights.transforms()
    test_transform = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_normalize.mean, std=imagenet_normalize.std),
    ])
    dataset = BreastTumorDataset(test_rows, transform=test_transform)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

In [ ]:
def build_densenet121():
    weights = DenseNet121_Weights.DEFAULT
    model = models.densenet121(weights=weights)
    num_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.25),
        nn.Linear(num_features, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(p=0.15),
        nn.Linear(256, 2),
    )
    return model, weights


def build_vgg16():
    weights = VGG16_Weights.DEFAULT
    model = models.vgg16(weights=weights)
    num_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.25),
        nn.Linear(num_features, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(p=0.15),
        nn.Linear(256, 2),
    )
    return model, weights


def build_resnet50():
    weights = ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.25),
        nn.Linear(num_features, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(p=0.15),
        nn.Linear(256, 2),
    )
    return model, weights


MODEL_BUILDERS = {
    'DenseNet121': build_densenet121,
    'VGG16': build_vgg16,
    'ResNet50': build_resnet50,
}

In [ ]:
def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ('model_state_dict', 'state_dict'):
            if key in checkpoint:
                return checkpoint[key]
    return checkpoint


def load_trained_model(model_name):
    checkpoint_path = CHECKPOINTS[model_name]
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f'{model_name} checkpoint not found at {checkpoint_path}. '
            'Save the trained model there or update CHECKPOINTS.'
        )

    model, weights = MODEL_BUILDERS[model_name]()
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(extract_state_dict(checkpoint))
    model = model.to(DEVICE)
    model.eval()
    return model, weights


def predict_positive_probabilities(model, data_loader):
    all_labels = []
    all_probabilities = []
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(DEVICE, non_blocking=True)
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)
            all_probabilities.extend(probabilities[:, POSITIVE_CLASS_ID].cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_probabilities)

In [ ]:
def roc_curve_and_auc(y_true, y_score, positive_label=1):
    y_true = (np.asarray(y_true) == positive_label).astype(int)
    y_score = np.asarray(y_score)
    order = np.argsort(-y_score)
    y_true = y_true[order]
    y_score = y_score[order]

    positives = y_true.sum()
    negatives = len(y_true) - positives
    if positives == 0 or negatives == 0:
        raise ValueError('ROC AUC requires both positive and negative test examples.')

    distinct_indices = np.where(np.diff(y_score))[0]
    threshold_indices = np.r_[distinct_indices, y_true.size - 1]
    true_positives = np.cumsum(y_true)[threshold_indices]
    false_positives = 1 + threshold_indices - true_positives

    tpr = np.r_[0, true_positives / positives, 1]
    fpr = np.r_[0, false_positives / negatives, 1]
    auc = np.trapz(tpr, fpr)
    return fpr, tpr, float(auc)


roc_results = {}
for model_name in MODEL_BUILDERS:
    model, weights = load_trained_model(model_name)
    test_loader = build_test_loader(weights)
    y_true, y_score = predict_positive_probabilities(model, test_loader)
    fpr, tpr, auc = roc_curve_and_auc(y_true, y_score, positive_label=POSITIVE_CLASS_ID)
    roc_results[model_name] = {'fpr': fpr, 'tpr': tpr, 'auc': auc}
    print(f'{model_name} AUC ({CLASS_NAMES[POSITIVE_CLASS_ID]} positive class): {auc:.4f}')

In [ ]:
plt.figure(figsize=(8, 6))
for model_name, result in roc_results.items():
    plt.plot(result['fpr'], result['tpr'], linewidth=2, label=f'{model_name} (AUC = {result["auc"]:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1, label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC Curves ({CLASS_NAMES[POSITIVE_CLASS_ID]} as Positive Class)')
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.grid(alpha=0.25)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()